# NB1.5: S and D Orbital Fraction Descriptors

**Goal:** Extract s-orbital and d-orbital fractions at VBM and CBM,
complementing the p-orbital fractions from nb1.

**Method:** Same as nb1: integrate orbital-projected DOS within a 1.0 eV
window around VBM/CBM, normalize by total DOS.

**Output:** CSV with uid, E_sfrac_VBM, E_sfrac_CBM, E_dfrac_VBM, E_dfrac_CBM

**Location:** `Keshav-DDP/Weight-contribution/contribution-model/nb1.5_sd_orbital_fraction.ipynb`

## Cell 1: Configuration

In [1]:
import os

# =============================================================================
# PATHS -- CHANGE THESE IF YOU MOVE THE NOTEBOOK
# =============================================================================

# This notebook lives in: Keshav-DDP/Weight-contribution/contribution-model/
BASE_DIR = os.path.abspath(os.path.join("..", ".."))

# Rashba compound folders with dos_*_dos.dat files
VASP_DIR = os.path.join(BASE_DIR, "Inverse-design", "rashba")

# Rashba CSV (to get uid list and bandgap for VBM/CBM location)
RASHBA_CSV = os.path.join(BASE_DIR, "Data", "rashba.csv")

# Existing p-fraction CSV (to merge with and verify alignment)
PFRAC_CSV = os.path.join(".", "best-descriptors-model-study",
                          "best_combo_0_w10_radius_mean.csv")

# Output
RESULTS_DIR = os.path.join(".", "nb1.5_sd_fraction-results")
OUTPUT_CSV = os.path.join(RESULTS_DIR, "sd_orbital_fractions.csv")

os.makedirs(RESULTS_DIR, exist_ok=True)

# Energy window (eV) around VBM/CBM to integrate
WINDOW = 1.0  # Same as best window from nb1

# =============================================================================
print("=" * 65)
print("  PATH CONFIGURATION")
print("=" * 65)
for name, path in [("BASE_DIR", BASE_DIR), ("VASP_DIR", VASP_DIR),
                    ("RASHBA_CSV", RASHBA_CSV), ("PFRAC_CSV", PFRAC_CSV),
                    ("OUTPUT_CSV", OUTPUT_CSV)]:
    exists = os.path.exists(path) if name != "OUTPUT_CSV" else "will create"
    status = "OK" if exists == True else ("OUTPUT" if exists == "will be created" else "MISSING")
    print(f"  [{status:7s}] {name:15s} = {path}")

  PATH CONFIGURATION
  [OK     ] BASE_DIR        = c:\Users\AbCMS_Lab\Desktop\Keshav-DDP
  [OK     ] VASP_DIR        = c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\rashba
  [OK     ] RASHBA_CSV      = c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Data\rashba.csv
  [OK     ] PFRAC_CSV       = .\best-descriptors-model-study\best_combo_0_w10_radius_mean.csv
  [MISSING] OUTPUT_CSV      = .\nb1.5_sd_fraction-results\sd_orbital_fractions.csv


## Cell 2: Imports

In [2]:
import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

print("Imports OK.")

Imports OK.


## Cell 3: Load rashba.csv and build uid -> folder mapping

In [3]:
df = pd.read_csv(RASHBA_CSV)
print(f"Loaded rashba.csv: {df.shape[0]} rows, {df['uid'].nunique()} unique UIDs")

# Get unique UIDs with their bandgap
uid_info = df.groupby('uid').agg({
    'Formula': 'first',
    'bandgap': 'first',
}).reset_index()
print(f"Unique compounds: {len(uid_info)}")


def find_compound_folder(uid, vasp_dir):
    """Find folder for a given uid."""
    for folder in glob.glob(os.path.join(vasp_dir, "*")):
        folder_name = os.path.basename(folder)
        idx = folder_name.rfind('-')
        if idx != -1 and folder_name[idx+1:] == uid:
            return folder
    return None


# Build mapping
uid_to_folder = {}
for _, row in uid_info.iterrows():
    folder = find_compound_folder(row['uid'], VASP_DIR)
    if folder:
        uid_to_folder[row['uid']] = folder

print(f"Found folders for {len(uid_to_folder)} / {len(uid_info)} compounds")

Loaded rashba.csv: 205 rows, 99 unique UIDs
Unique compounds: 99
Found folders for 99 / 99 compounds


## Cell 4: DOS parsing and orbital fraction extraction

In [4]:
def load_dos_files(compound_folder):
    """
    Load all dos_*_dos.dat files for a compound.
    Each file has columns: energy, s, p, d
    Returns dict: {element: DataFrame}
    """
    dos_files = glob.glob(os.path.join(compound_folder, "dos_*_dos.dat"))
    element_dos = {}

    for f in dos_files:
        # Extract element name from filename: dos_As_dos.dat -> As
        basename = os.path.basename(f)
        parts = basename.replace("dos_", "").replace("_dos.dat", "")
        element = parts

        try:
            data = np.loadtxt(f, skiprows=1)  # skip header line
            dos_df = pd.DataFrame(data, columns=['energy', 's', 'p', 'd'])
            element_dos[element] = dos_df
        except Exception as e:
            print(f"  Error loading {f}: {e}")

    return element_dos


def compute_orbital_fractions(element_dos, bandgap, window=1.0):
    """
    Compute s, p, d orbital fractions at VBM and CBM.

    VBM region: [VBM - window, VBM] where VBM = 0 (Fermi level)
    CBM region: [CBM, CBM + window] where CBM = bandgap

    Fraction = sum(orbital) / sum(all orbitals) in the window.
    """
    result = {
        'E_sfrac_VBM': np.nan, 'E_pfrac_VBM': np.nan, 'E_dfrac_VBM': np.nan,
        'E_sfrac_CBM': np.nan, 'E_pfrac_CBM': np.nan, 'E_dfrac_CBM': np.nan,
    }

    if not element_dos:
        return result

    # Stack all elements: sum DOS across elements for each orbital
    ref_energy = list(element_dos.values())[0]['energy'].values

    total_s = np.zeros_like(ref_energy)
    total_p = np.zeros_like(ref_energy)
    total_d = np.zeros_like(ref_energy)

    for elem, dos_df in element_dos.items():
        total_s += dos_df['s'].values
        total_p += dos_df['p'].values
        total_d += dos_df['d'].values

    total_all = total_s + total_p + total_d

    # VBM window: energy in [-window, 0]
    vbm_mask = (ref_energy >= -window) & (ref_energy <= 0)
    # CBM window: energy in [bandgap, bandgap + window]
    cbm_mask = (ref_energy >= bandgap) & (ref_energy <= bandgap + window)

    for label, mask in [('VBM', vbm_mask), ('CBM', cbm_mask)]:
        s_sum = total_s[mask].sum()
        p_sum = total_p[mask].sum()
        d_sum = total_d[mask].sum()
        all_sum = total_all[mask].sum()

        if all_sum > 1e-10:
            result[f'E_sfrac_{label}'] = s_sum / all_sum
            result[f'E_pfrac_{label}'] = p_sum / all_sum
            result[f'E_dfrac_{label}'] = d_sum / all_sum
        else:
            result[f'E_sfrac_{label}'] = 0.0
            result[f'E_pfrac_{label}'] = 0.0
            result[f'E_dfrac_{label}'] = 0.0

    return result


# Test on one compound
test_uid = uid_info.iloc[0]['uid']
if test_uid in uid_to_folder:
    test_dos = load_dos_files(uid_to_folder[test_uid])
    test_bg = uid_info.iloc[0]['bandgap']
    test_result = compute_orbital_fractions(test_dos, test_bg, window=WINDOW)
    formula = uid_info.iloc[0]['Formula']
    print(f"Test: {formula} (bandgap={test_bg:.3f} eV)")
    print(f"  Elements found: {list(test_dos.keys())}")
    for k, v in test_result.items():
        print(f"  {k}: {v:.4f}")

  Error loading c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\rashba\SSeW-001e03f2c095\dos_total_dos.dat: Shape of passed values is (301, 2), indices imply (301, 4)
Test: SSeW (bandgap=1.417 eV)
  Elements found: ['Se', 'S', 'W']
  E_sfrac_VBM: 0.0187
  E_pfrac_VBM: 0.2579
  E_dfrac_VBM: 0.7234
  E_sfrac_CBM: 0.0302
  E_pfrac_CBM: 0.2127
  E_dfrac_CBM: 0.7571


## Cell 5: Extract for all compounds

In [5]:
print("=" * 65)
print("  EXTRACTING ORBITAL FRACTIONS")
print("=" * 65)

all_results = []
errors = []

for i, row in uid_info.iterrows():
    uid = row['uid']
    formula = row['Formula']
    bandgap = row['bandgap']

    if uid not in uid_to_folder:
        errors.append((uid, formula, "folder not found"))
        continue

    try:
        element_dos = load_dos_files(uid_to_folder[uid])
        fracs = compute_orbital_fractions(element_dos, bandgap, window=WINDOW)
        fracs['uid'] = uid
        fracs['Formula'] = formula
        all_results.append(fracs)
    except Exception as e:
        errors.append((uid, formula, str(e)))

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(uid_info)} compounds...")

df_result = pd.DataFrame(all_results)
print(f"\nExtracted: {len(df_result)} compounds")
if errors:
    print(f"Errors: {len(errors)}")
    for e in errors[:5]:
        print(f"  {e}")

  EXTRACTING ORBITAL FRACTIONS
  Error loading c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\rashba\SSeW-001e03f2c095\dos_total_dos.dat: Shape of passed values is (301, 2), indices imply (301, 4)
  Error loading c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\rashba\Sn2Te2-03bcf7dcdaf2\dos_total_dos.dat: Shape of passed values is (301, 2), indices imply (301, 4)
  Error loading c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\rashba\ClSbTe-04fdd7d1ec5c\dos_total_dos.dat: Shape of passed values is (301, 2), indices imply (301, 4)
  Error loading c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\rashba\WMo3Se8-05a06afa3b20\dos_total_dos.dat: Shape of passed values is (301, 2), indices imply (301, 4)
  Error loading c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\rashba\CrW3Se8-0b7696e1f4c9\dos_total_dos.dat: Shape of passed values is (301, 2), indices imply (301, 4)
  Error loading c:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\rashba\ClSbSe-0c0fbdaf8f4a\dos

## Cell 6: Verify against existing p-fraction

In [6]:
if os.path.exists(PFRAC_CSV):
    df_pfrac = pd.read_csv(PFRAC_CSV)
    print("Verifying against existing p-fraction CSV:")
    print(f"  Existing: {df_pfrac.shape[0]} rows")

    # Merge and compare
    merged = df_result.merge(df_pfrac[['uid', 'E_pfrac_VBM', 'E_pfrac_CBM']],
                              on='uid', how='inner', suffixes=('_new', '_old'))

    if 'E_pfrac_VBM_new' in merged.columns and 'E_pfrac_VBM_old' in merged.columns:
        corr_vbm = merged['E_pfrac_VBM_new'].corr(merged['E_pfrac_VBM_old'])
        corr_cbm = merged['E_pfrac_CBM_new'].corr(merged['E_pfrac_CBM_old'])
        print(f"  p-fraction VBM correlation (new vs old): {corr_vbm:.4f}")
        print(f"  p-fraction CBM correlation (new vs old): {corr_cbm:.4f}")
        if corr_vbm > 0.99:
            print("  GOOD: p-fractions match!")
        else:
            print("  WARNING: p-fractions differ. Check window or integration method.")
else:
    print(f"No existing p-fraction CSV found at {PFRAC_CSV}. Skipping verification.")

Verifying against existing p-fraction CSV:
  Existing: 99 rows
  p-fraction VBM correlation (new vs old): 0.9939
  p-fraction CBM correlation (new vs old): 0.9995
  GOOD: p-fractions match!


## Cell 7: Summary statistics

In [7]:
print("=" * 65)
print("  ORBITAL FRACTION STATISTICS")
print("=" * 65)

orbital_cols = ['E_sfrac_VBM', 'E_pfrac_VBM', 'E_dfrac_VBM',
                'E_sfrac_CBM', 'E_pfrac_CBM', 'E_dfrac_CBM']

for col in orbital_cols:
    vals = df_result[col].dropna()
    print(f"\n  {col}:")
    print(f"    mean={vals.mean():.4f}, std={vals.std():.4f}, "
          f"min={vals.min():.4f}, max={vals.max():.4f}")
    # How many compounds have >10% of this orbital?
    n_sig = (vals > 0.1).sum()
    print(f"    Compounds with >{col}>0.1: {n_sig}/{len(vals)} ({n_sig/len(vals)*100:.1f}%)")

# Check: s + p + d should = 1.0
df_result['sum_VBM'] = df_result['E_sfrac_VBM'] + df_result['E_pfrac_VBM'] + df_result['E_dfrac_VBM']
df_result['sum_CBM'] = df_result['E_sfrac_CBM'] + df_result['E_pfrac_CBM'] + df_result['E_dfrac_CBM']
print(f"\n  Sanity check (s+p+d should = 1.0):")
print(f"    VBM sum: mean={df_result['sum_VBM'].mean():.4f}, min={df_result['sum_VBM'].min():.4f}, max={df_result['sum_VBM'].max():.4f}")
print(f"    CBM sum: mean={df_result['sum_CBM'].mean():.4f}, min={df_result['sum_CBM'].min():.4f}, max={df_result['sum_CBM'].max():.4f}")

  ORBITAL FRACTION STATISTICS

  E_sfrac_VBM:
    mean=0.0603, std=0.0564, min=0.0077, max=0.2633
    Compounds with >E_sfrac_VBM>0.1: 20/99 (20.2%)

  E_pfrac_VBM:
    mean=0.6674, std=0.2816, min=0.2146, max=0.9451
    Compounds with >E_pfrac_VBM>0.1: 99/99 (100.0%)

  E_dfrac_VBM:
    mean=0.2723, std=0.3167, min=0.0081, max=0.7683
    Compounds with >E_dfrac_VBM>0.1: 45/99 (45.5%)

  E_sfrac_CBM:
    mean=0.0594, std=0.0561, min=0.0091, max=0.2921
    Compounds with >E_sfrac_CBM>0.1: 19/99 (19.2%)

  E_pfrac_CBM:
    mean=0.5565, std=0.3597, min=0.1064, max=0.9401
    Compounds with >E_pfrac_CBM>0.1: 99/99 (100.0%)

  E_dfrac_CBM:
    mean=0.3841, std=0.3926, min=0.0056, max=0.8770
    Compounds with >E_dfrac_CBM>0.1: 45/99 (45.5%)

  Sanity check (s+p+d should = 1.0):
    VBM sum: mean=1.0000, min=1.0000, max=1.0000
    CBM sum: mean=1.0000, min=1.0000, max=1.0000


## Cell 8: Save

In [8]:
# Drop verification columns
save_cols = ['uid', 'Formula', 'E_sfrac_VBM', 'E_pfrac_VBM', 'E_dfrac_VBM',
             'E_sfrac_CBM', 'E_pfrac_CBM', 'E_dfrac_CBM']
df_save = df_result[save_cols]
df_save.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
print(f"  {len(df_save)} compounds, {len(save_cols)} columns")

print(f"""
  Features extracted:
    E_sfrac_VBM: s-orbital fraction at VBM (1.0 eV window)
    E_pfrac_VBM: p-orbital fraction at VBM (verification, should match nb1)
    E_dfrac_VBM: d-orbital fraction at VBM
    E_sfrac_CBM: s-orbital fraction at CBM
    E_pfrac_CBM: p-orbital fraction at CBM
    E_dfrac_CBM: d-orbital fraction at CBM

  Next: Merge with nb7 and test if d-fraction improves regression/classification.
""")

Saved: .\nb1.5_sd_fraction-results\sd_orbital_fractions.csv
  99 compounds, 8 columns

  Features extracted:
    E_sfrac_VBM: s-orbital fraction at VBM (1.0 eV window)
    E_pfrac_VBM: p-orbital fraction at VBM (verification, should match nb1)
    E_dfrac_VBM: d-orbital fraction at VBM
    E_sfrac_CBM: s-orbital fraction at CBM
    E_pfrac_CBM: p-orbital fraction at CBM
    E_dfrac_CBM: d-orbital fraction at CBM

  Next: Merge with nb7 and test if d-fraction improves regression/classification.

